<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.3-rag-engine/practice/GCP_Capstone_4.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 4.3 — Vertex AI RAG Engine

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install the SDKs, authenticate with Application Default Credentials, and initialize the Vertex RAG module plus the `google-genai` client. Run this cell first — every exercise below depends on `rag`, `client`, `types`, and `PROJECT_ID`.

In [ ]:
!pip install -q google-cloud-aiplatform google-genai
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'us-central1'           # serverless RAG corpora (Vector Search 2.0) are us-central1-only

from vertexai import rag
import vertexai
from google import genai
from google.genai import types

vertexai.init(project=PROJECT_ID, location=LOCATION)
client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # Gemini 3.x generation: global (corpus/LOCATION stays regional)
print('Setup complete.')

## Prepare a new project for serverless RAG Engine (one-time)
A brand-new project needs three things before `create_corpus` works, all handled by the cell below (idempotent — safe to re-run):
1. **APIs enabled** — `vectorsearch.googleapis.com` backs the serverless vector store and is **off by default**; a missing enable is exactly what makes `create_corpus` fail with *“Vector Search API … has not been used … or it is disabled”*. (`aiplatform` + `storage` are enabled too.)
2. **Serverless mode** — the default “Scaled”/Spanner store is allowlist-only for new projects; Serverless mode is open to everyone.
3. **Propagation** — API enablement is eventually consistent, so the first corpus is created through `create_corpus_ready(...)`, which retries while enablement propagates.

> Serverless RAG corpora (Vector Search 2.0) are **`us-central1`-only** — not `asia-south1`. The corpus, embedding and retrieval client stays regional; only the Gemini-3.x generation client uses `global`.

In [ ]:
# One-time new-project prep for serverless RAG Engine. Idempotent — safe to re-run.
import google.auth, google.auth.transport.requests, requests, subprocess, time
from google.api_core import exceptions as _gexc
_RAG_LOCATION = 'us-central1'   # serverless RAG corpora (Vector Search 2.0) are us-central1-only

# 1) Enable the APIs a fresh project needs. vectorsearch backs the serverless vector store
#    (off by default -> the create_corpus 403); storage is for GCS import.
subprocess.run(['gcloud', 'services', 'enable',
                'aiplatform.googleapis.com', 'vectorsearch.googleapis.com',
                'storage.googleapis.com', '--project', PROJECT_ID], check=False)

# 2) Switch RAG Engine to Serverless mode (open to everyone, no allowlist).
_creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
_creds.refresh(google.auth.transport.requests.Request())
_r = requests.patch(
    f'https://{_RAG_LOCATION}-aiplatform.googleapis.com/v1beta1/'
    f'projects/{PROJECT_ID}/locations/{_RAG_LOCATION}/ragEngineConfig',
    headers={'Authorization': f'Bearer {_creds.token}'},
    json={'ragManagedDbConfig': {'serverless': {}}}, timeout=60)
print('Serverless mode:', 'ready' if _r.ok else f'{_r.status_code} {_r.text[:150]}')

# 3) create_corpus wrapper that waits out API-enablement propagation on fresh projects.
def create_corpus_ready(**kwargs):
    for _attempt in range(8):  # ~ up to 4 min
        try:
            return rag.create_corpus(**kwargs)
        except (_gexc.PermissionDenied, _gexc.FailedPrecondition) as e:
            m = str(e).lower()
            if 'has not been used' in m or 'is disabled' in m or 'service_disabled' in m:
                print('vectorsearch API still propagating; retrying in 30s...'); time.sleep(30); continue
            raise
    raise RuntimeError('vectorsearch.googleapis.com still not ready — wait 1-2 min and re-run.')
print('APIs enabled. First corpus uses create_corpus_ready(...) to ride out enablement propagation.')

## Create a docs bucket (named from your project)
The exercises import from Cloud Storage. This creates a bucket `{PROJECT_ID}-rag-docs` in `us-central1` and seeds sample files under `docs/`, `engineering/`, and `product/`, so every import exercise runs out of the box. Idempotent — safe to re-run.

In [ ]:
# Create a GCS bucket for this lab's documents (named from the project so it is globally
# unique) and seed sample files for the exercises. Idempotent — safe to re-run.
import subprocess
BUCKET = f'{PROJECT_ID}-rag-docs'
_exists = subprocess.run(['gcloud', 'storage', 'buckets', 'describe', f'gs://{BUCKET}',
                          '--project', PROJECT_ID], capture_output=True, text=True).returncode == 0
if not _exists:
    subprocess.run(['gcloud', 'storage', 'buckets', 'create', f'gs://{BUCKET}',
                    '--project', PROJECT_ID, '--location', 'us-central1',
                    '--uniform-bucket-level-access'], check=True)
_seed = {
    'docs/sample.txt': 'Retrieval-Augmented Generation (RAG) grounds an LLM in your own documents: chunk, embed, retrieve, then answer with citations. Vertex AI RAG Engine manages this for you.',
    'engineering/eng.txt': 'DocuMind engineering: the ingestion service chunks PDFs, embeds chunks with text-embedding-005, and writes vectors to the managed store. Retrieval uses nearest-neighbour search.',
    'product/prod.txt': 'DocuMind product: users upload documents and ask questions; the assistant answers with citations to the source passages. Pricing is per document processed and per query.',
}
for _path, _text in _seed.items():
    _local = _path.split('/')[-1]
    with open(_local, 'w') as _f:
        _f.write(_text)
    subprocess.run(['gcloud', 'storage', 'cp', _local, f'gs://{BUCKET}/{_path}',
                    '--project', PROJECT_ID], check=True)
print(f'Docs bucket ready: gs://{BUCKET}/  (seeded docs/, engineering/, product/)')

## Exercise 1: Create a Corpus

**Difficulty:** Easy

Create a RAG corpus with text-embedding-005. Print the resource name.

1. Configure RagEmbeddingModelConfig
2. Call rag.create_corpus()
3. Print corpus.name

In [ ]:
embedding_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model='publishers/google/models/text-embedding-005'
    )
)

corpus = create_corpus_ready(
    display_name='documind-lesson43',
    description='Lesson 4.3 test corpus',
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_config),
)
print(f'Corpus: {corpus.name}')

# List corpora
for c in rag.list_corpora():
    print(f'  {c.display_name}: {c.name}')

## Exercise 2: Import from GCS

**Difficulty:** Easy

Upload a PDF to GCS. Import into corpus. Print imported file count.

1. gsutil cp file.pdf gs://bucket/docs/
2. rag.import_files() with chunk_size=512
3. Check imported_rag_files_count

In [ ]:
%%bash
# The Setup "docs bucket" cell already created and seeded gs://<project>-rag-docs/.
# List what's there (add your own files with: gcloud storage cp FILE gs://<project>-rag-docs/docs/).
gcloud storage ls "gs://$(gcloud config get-value project 2>/dev/null)-rag-docs/docs/"

In [ ]:
# Grant the RAG service agent read on the docs bucket (create_corpus above provisioned
# it); without this grant, import returns 0 files silently.
import subprocess
_pn = subprocess.run(['gcloud', 'projects', 'describe', PROJECT_ID, '--format=value(projectNumber)'],
                     capture_output=True, text=True).stdout.strip()
subprocess.run(['gcloud', 'storage', 'buckets', 'add-iam-policy-binding', f'gs://{BUCKET}',
                '--member', f'serviceAccount:service-{_pn}@gcp-sa-vertex-rag.iam.gserviceaccount.com',
                '--role', 'roles/storage.objectViewer', '--project', PROJECT_ID], check=False)

response = rag.import_files(
    corpus.name,
    [f'gs://{BUCKET}/docs/'],
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512, chunk_overlap=100)),
    max_embedding_requests_per_min=900,
)
print(f'Imported: {response.imported_rag_files_count}')
print(f'Skipped: {response.skipped_rag_files_count}')
if response.imported_rag_files_count == 0:
    print('WARNING: 0 files imported. The service-agent grant above may still be '
          'propagating (~1-2 min) — wait and re-run this cell.')

## Exercise 3: Direct Retrieval

**Difficulty:** Easy

Use retrieval_query() to search corpus. Print top-5 chunks with scores.

1. Call rag.retrieval_query()
2. Loop response.contexts.contexts
3. Print source_uri, score, text preview

In [ ]:
response = rag.retrieval_query(
    rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
    text='What is RAG?',
    rag_retrieval_config=rag.RagRetrievalConfig(
        top_k=5,
        filter=rag.Filter(vector_distance_threshold=0.5)),
)

for ctx in response.contexts.contexts:
    print(f'Source: {ctx.source_uri}')
    print(f'Score: {ctx.score:.3f}, Distance: {ctx.distance:.3f}')
    print(f'Text: {ctx.text[:150]}...\n')

## Exercise 4: Grounded Generation

**Difficulty:** Medium

Use types.Tool(retrieval=...) + generate_content(). Print answer + grounding citations.

1. Create RAG tool from corpus
2. Pass to GenerativeModel
3. Print grounding_chunks and grounding_supports

In [ ]:
rag_retrieval_tool = types.Tool(
    retrieval=types.Retrieval(
        vertex_rag_store=types.VertexRagStore(
            rag_resources=[types.VertexRagStoreRagResource(rag_corpus=corpus.name)],
            rag_retrieval_config=types.RagRetrievalConfig(
                top_k=5,
                filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Summarize the main topics in my documents',
    config=types.GenerateContentConfig(tools=[rag_retrieval_tool]))
print(response.text)

# Grounding metadata — automatic citations
for candidate in response.candidates:
    gm = candidate.grounding_metadata
    if gm and gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            print(f'  Source: {chunk.retrieved_context.uri}')
    if gm and gm.grounding_supports:
        for support in gm.grounding_supports:
            print(f'  Claim: {support.segment.text}')
            print(f'  Backed by: {support.grounding_chunk_indices}')

## Exercise 5: Drive Connector

**Difficulty:** Medium

Share a Drive folder with the RAG service account. Import and query.

1. Find RAG service account in IAM
2. Share Drive folder as Viewer
3. Import with Drive URL path

In [ ]:
%%bash
# Step 1: Find the RAG service account to share your Drive folder with.
# The RAG Engine uses the Vertex AI RAG data service agent.
# NOTE: %%bash runs a subshell — the Python PROJECT_ID is NOT visible here,
# so resolve the active project from gcloud config instead.
PROJECT_ID=$(gcloud config get-value project 2>/dev/null)
gcloud projects get-iam-policy "$PROJECT_ID" \
  --flatten='bindings[].members' \
  --format='table(bindings.members)' \
  --filter='bindings.members:rag' 2>/dev/null || true

# Typical form:
#   service-<PROJECT_NUMBER>@gcp-sa-vertex-rag.iam.gserviceaccount.com
# Step 2 (manual): In Google Drive, share the target folder as Viewer
#   with that service account email.

In [ ]:
# Step 3: Import from a shared Drive folder, then retrieve. Drive folders are your own, so
# this is bring-your-own: share your folder (Viewer) with the RAG service agent from Step 1,
# copy its id from the URL, set DRIVE_FOLDER_ID, and re-run.
DRIVE_FOLDER_ID = 'YOUR_FOLDER_ID'  # CHANGE, or leave as-is to skip
if DRIVE_FOLDER_ID == 'YOUR_FOLDER_ID':
    print('Skipping Drive import — set DRIVE_FOLDER_ID to your own folder id (shared with the '
          'service agent from Step 1) to run this exercise.')
else:
    drive_response = rag.import_files(
        corpus.name,
        [f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}'],
        transformation_config=rag.TransformationConfig(
            chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100)),
        max_embedding_requests_per_min=900)
    print(f'Drive import: {drive_response.imported_rag_files_count}')

    # Confirm Drive-sourced chunks come back
    drive_hits = rag.retrieval_query(
        rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
        text='What do the Drive documents cover?',
        rag_retrieval_config=rag.RagRetrievalConfig(top_k=5))
    for ctx in drive_hits.contexts.contexts:
        print(f'  {ctx.source_uri}  (score {ctx.score:.3f})')

## Exercise 6: Layout Parser Import

**Difficulty:** Medium

Import PDFs with LayoutParserConfig. Compare chunk quality vs default.

1. Pass layout_parser=rag.LayoutParserConfig(...)
2. Query same question with and without Layout Parser
3. Compare chunk coherence

In [ ]:
# PREREQUISITE: this exercise needs a Document AI Layout-Parser processor id (replace
# YOUR_LAYOUT_PROCESSOR_ID below); it reuses the docs bucket seeded in Setup.
# Layout-aware parsing keeps headings, tables and paragraphs intact instead of
# splitting on raw token counts. Import the SAME source into a fresh corpus so
# you can compare chunk coherence side by side.

layout_corpus = rag.create_corpus(
    display_name='documind-lesson43-layout',
    description='Layout Parser comparison corpus',
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_config),
)
print(f'Layout corpus: {layout_corpus.name}')

rag.import_files(
    layout_corpus.name,
    [f'gs://{BUCKET}/docs/'],  # same source as Exercise 2
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100)),
    layout_parser=rag.LayoutParserConfig(
        processor_name=f'projects/{PROJECT_ID}/locations/us/processors/YOUR_LAYOUT_PROCESSOR_ID',
        max_parsing_requests_per_min=120,
    ),
    max_embedding_requests_per_min=900,
)

question = 'What is RAG?'
print('--- Default chunking ---')
for ctx in rag.retrieval_query(
        rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
        text=question,
        rag_retrieval_config=rag.RagRetrievalConfig(top_k=3)).contexts.contexts:
    print(f'  {ctx.text[:180]}...')

print('\n--- Layout Parser chunking ---')
for ctx in rag.retrieval_query(
        rag_resources=[rag.RagResource(rag_corpus=layout_corpus.name)],
        text=question,
        rag_retrieval_config=rag.RagRetrievalConfig(top_k=3)).contexts.contexts:
    print(f'  {ctx.text[:180]}...')

## Exercise 7: Multi-Corpus Search

**Difficulty:** Challenge

Create 2 corpora (engineering + product). Import different docs. Query across both.

1. Create corpus_eng and corpus_prod
2. Import different document sets into each
3. Pass both as RagResource objects in one query

In [ ]:
# Create two topic-specific corpora sharing the same embedding config.
corpus_eng = rag.create_corpus(
    display_name='documind-eng',
    description='Engineering docs',
    backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=embedding_config))
corpus_prod = rag.create_corpus(
    display_name='documind-product',
    description='Product docs',
    backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=embedding_config))
print(f'Eng:     {corpus_eng.name}')
print(f'Product: {corpus_prod.name}')

tx = rag.TransformationConfig(
    chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100))

rag.import_files(corpus_eng.name, [f'gs://{BUCKET}/engineering/'],
                 transformation_config=tx, max_embedding_requests_per_min=900)
rag.import_files(corpus_prod.name, [f'gs://{BUCKET}/product/'],
                 transformation_config=tx, max_embedding_requests_per_min=900)

# retrieval_query supports only ONE corpus per call, so query each corpus
# separately and merge the contexts by distance (closest first).
merged = []
for rc in (corpus_eng, corpus_prod):
    resp = rag.retrieval_query(
        rag_resources=[rag.RagResource(rag_corpus=rc.name)],
        text='How does the product handle document ingestion?',
        rag_retrieval_config=rag.RagRetrievalConfig(
            top_k=6, filter=rag.Filter(vector_distance_threshold=0.5)))
    merged.extend(resp.contexts.contexts)
merged.sort(key=lambda c: c.distance)
for ctx in merged[:6]:
    print(f'  {ctx.source_uri}  (score {ctx.score:.3f})')

## Exercise 8: ManagedRAG Module

**Difficulty:** Challenge

Build complete ManagedRAG class with create_corpus(), ingest(), retrieve(), ask().

1. Implement all 4 methods
2. Test end-to-end: create → ingest → ask
3. Print grounding metadata from ask()

In [ ]:
class ManagedRAG:
    def __init__(self, project, location='us-central1'):
        vertexai.init(project=project, location=location)
        self.client = genai.Client(enterprise=True, project=project, location='global')  # generation: global (corpus stays regional)
        self.corpus = None

    def create_corpus(self, name, description=''):
        emb = rag.RagEmbeddingModelConfig(
            vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
                publisher_model='publishers/google/models/text-embedding-005'))
        self.corpus = rag.create_corpus(
            display_name=name, description=description,
            backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=emb))
        return self.corpus.name

    def use_corpus(self, corpus_name):
        self.corpus = rag.get_corpus(name=corpus_name)

    def ingest(self, paths, chunk_size=512, chunk_overlap=100):
        return rag.import_files(
            self.corpus.name, paths,
            transformation_config=rag.TransformationConfig(
                rag.ChunkingConfig(chunk_size=chunk_size, chunk_overlap=chunk_overlap)),
            max_embedding_requests_per_min=900)

    def retrieve(self, query, top_k=5):
        return rag.retrieval_query(
            rag_resources=[rag.RagResource(rag_corpus=self.corpus.name)],
            text=query,
            rag_retrieval_config=rag.RagRetrievalConfig(
                top_k=top_k, filter=rag.Filter(vector_distance_threshold=0.5)))

    def ask(self, question, model_name='gemini-3.6-flash'):
        rag_tool = types.Tool(retrieval=types.Retrieval(
            vertex_rag_store=types.VertexRagStore(
                rag_resources=[types.VertexRagStoreRagResource(rag_corpus=self.corpus.name)],
                rag_retrieval_config=types.RagRetrievalConfig(
                    top_k=5, filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))
        return self.client.models.generate_content(
            model=model_name, contents=question,
            config=types.GenerateContentConfig(tools=[rag_tool]))

print('ManagedRAG class ready')

In [ ]:
# End-to-end test: create -> ingest -> ask, then print grounding metadata.
mrag = ManagedRAG(PROJECT_ID, LOCATION)
mrag.create_corpus('documind-managed-e2e', 'ManagedRAG end-to-end test')
mrag.ingest([f'gs://{BUCKET}/docs/'])

answer = mrag.ask('What are the key points in my documents?')
print(answer.text)

for candidate in answer.candidates:
    gm = candidate.grounding_metadata
    if gm and gm.grounding_chunks:
        print('\nCitations:')
        for chunk in gm.grounding_chunks:
            print(f'  Source: {chunk.retrieved_context.uri}')

## Cleanup (optional)

Delete the corpora you created to avoid ongoing storage costs. RAG Engine storage is billed per GB-month; leaving idle corpora around adds up. Uncomment to run.

In [ ]:
# for c in [corpus, layout_corpus, corpus_eng, corpus_prod, mrag.corpus]:
#     try:
#         rag.delete_corpus(name=c.name)
#         print(f'Deleted {c.name}')
#     except Exception as e:
#         print(f'Skip {c}: {e}')